In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import plotly
import plotly.offline as py
from plotly.graph_objs import *
plotly.offline.init_notebook_mode(connected=True)

# TOPIC 2

In [233]:
# load hiv data set and split into different years

hiv = pd.read_csv('hiv_data.csv').ix[1:,]
hiv.columns = ['country', 'hiv_cases', 'hiv_cases', 'hiv_cases', 'hiv_cases']

# 2015 data
hiv2015 = hiv.ix[:,0:2]
# 2010 data
hiv2010 = pd.concat([hiv['country'], hiv.ix[:,2]], axis=1)
# 2005 data
hiv2005 = pd.concat([hiv['country'], hiv.ix[:,3]], axis=1)
# 2000 data
hiv2000 = pd.concat([hiv['country'], hiv.ix[:,4]], axis=1)

In [234]:
# rename columns and load tb dataset

columns = ['country', 'g_whoregion', 'year', 'e_inc_num', 'e_inc_tbhiv_num', 'e_tbhiv_prct']
tb = pd.read_csv('basic_TB_data.csv')[columns]
tb.columns = ['country', 'region', 'year','all_tb', 'tb_and_hiv', 'tb_and_hiv_percent']

In [242]:
# merge tb and hiv dataset on country column
tb2015 = tb[tb['year'] == 2015].merge(hiv2015, on='country')
tb2010 = tb[tb['year'] == 2010].merge(hiv2010, on='country')
tb2005 = tb[tb['year'] == 2005].merge(hiv2005, on='country')
tb2000 = tb[tb['year'] == 2000].merge(hiv2000, on='country')

tbcombined = pd.concat([tb2015, tb2010, tb2005, tb2000], axis=0)

# only concerned with rows that have data in hiv cases
tbcombined = tbcombined[tbcombined['hiv_cases'] != 'No data']

# create new column: cases with only tb
tbcombined['only_tb'] = tbcombined['all_tb'] - tbcombined['tb_and_hiv']

# change hiv range brackets to number
def to_int(x):
    x = (x.split('[')[0][:-1]).replace(" ", "")
    if '&lt;' in x:
        x = x[4:]
    return int(x)
tbcombined['hiv_cases'] = tbcombined['hiv_cases'].apply(to_int)

tb2015 = tbcombined[tbcombined['year']==2015]
tb2010 = tbcombined[tbcombined['year']==2010]
tb2005 = tbcombined[tbcombined['year']==2005]
tb2000 = tbcombined[tbcombined['year']==2000]

# we see we have data for roughly 100 countries in each year
# print(len(tb2015))
# print(len(tb2010))
# print(len(tb2005))
# print(len(tb2000))

len(tbcombined)

421

In [240]:
tbcombined.head()

,country,region,year,all_tb,tb_and_hiv,tb_and_hiv_percent,hiv_cases,only_tb
0,Afghanistan,EMR,2015,61000,460.0,0.75,6900,60540.0
2,Algeria,AFR,2015,30000,200.0,0.68,8800,29800.0
4,Angola,AFR,2015,93000,28000.0,30.00,320000,65000.0
6,Argentina,AMR,2015,11000,790.0,7.10,110000,10210.0
7,Armenia,EUR,2015,1200,110.0,9.10,3600,1090.0


graphs

1. scatter plot: total hiv+tb vs total tb no hiv
2. bar graph: regions vs total hiv+tb and tb
3. bar graph: regions vs percentage of hiv+tb cases 
4. 


# GRAPH 1: HIV VS TB

In [96]:
# FOR USE WITH ALL GRAPHS

# data by years


In [168]:
# Create traces

from scipy import stats
slope, intercept, r_value, p_value, std_err = stats.linregress(tb2015['hiv_cases'],tb2015['tb_all_cases'])
line = slope*tb2015['hiv_cases']+intercept

trace1 = Scatter(
    x=tb2015['hiv_cases'], y=tb2015['tb_all_cases'],
    mode='markers',
    name='2015',
    marker=dict(color= 'rgb(0, 255, 255)')
)

trace2 = Scatter(
                  x=tb2015['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2015Line',
    marker=dict(color= 'rgb(0, 255, 255)')
)


slope, intercept, r_value, p_value, std_err = stats.linregress(tb2010['hiv_cases'],tb2010['tb_all_cases'])
line = slope*tb2010['hiv_cases']+intercept


trace3 = Scatter(
    x=tb2010['hiv_cases'], y=tb2010['tb_all_cases'],
    mode='markers',
    name='2010',
    marker=dict(color= 'rgb(250, 128, 114)')
)

trace4 = Scatter(
                  x=tb2010['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2010Line',
    marker=dict(color= 'rgb(250, 128, 114)')
)

slope, intercept, r_value, p_value, std_err = stats.linregress(tb2005['hiv_cases'],tb2005['tb_all_cases'])
line = slope*tb2005['hiv_cases']+intercept



trace5 = Scatter(
    x=tb2005['hiv_cases'], y=tb2005['tb_all_cases'],
    mode='markers',
    name='2005',
    marker=dict(color= 'rgb(142, 69, 133)')
)

trace6 = Scatter(
                  x=tb2005['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2005Line',
    marker=dict(color= 'rgb(142, 69, 133)')
)

slope, intercept, r_value, p_value, std_err = stats.linregress(tb2000['hiv_cases'],tb2000['tb_all_cases'])
line = slope*tb2000['hiv_cases']+intercept

trace7 = Scatter(
    x=tb2000['hiv_cases'], y=tb2000['tb_all_cases'],
    mode='markers',
    name='2000',
    marker=dict(color= 'rgb(218, 165, 32)')
)

trace8 = Scatter(
                  x=tb2000['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2000Line',
    marker=dict(color= 'rgb(218, 165, 32)')
)

data = Data([trace1, trace2, trace3, trace4, trace5, trace6, trace7, trace8])
layout = Layout(
    title='HIV Cases VS Tuberculosis Cases. Data source: WHO',
    yaxis=dict(title='Tuberculosis Cases'),
    xaxis=dict(title='HIV Cases'),
    updatemenus=list([
        dict(
            x=-0.05,
            y=1,
            yanchor='top',
            buttons=list([
                dict(
                    args=['visible', [True, True, True, True, True, True, True, True]],
                    label='All',
                    method='restyle'
                ),
                dict(
                    args=['visible', [True, True, False, False, False, False, False, False]],
                    label='2015',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, True, True, False, False, False, False]],
                    label='2000',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, True, True, False, False]],
                    label='2005',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, False, False, True, True]],
                    label='2000',
                    method='restyle'
                )
            ]),
        )
    ]),
)
fig = Figure(data=data, layout=layout)
py.iplot(fig)

### pearson correlation score. averaged throughout the years, we roughly have a moderate positive correlation

In [214]:
# correlation
print(tbcombined[['hiv_cases', 'tb_all_cases']].corr(), '\n')
print(tb2015[['hiv_cases', 'tb_all_cases']].corr(), '\n')
print(tb2010[['hiv_cases', 'tb_all_cases']].corr(), '\n')
print(tb2005[['hiv_cases', 'tb_all_cases']].corr(), '\n')
print(tb2000[['hiv_cases', 'tb_all_cases']].corr(), '\n')

              hiv_cases  tb_all_cases
hiv_cases      1.000000      0.382508
tb_all_cases   0.382508      1.000000 

              hiv_cases  tb_all_cases
hiv_cases      1.000000      0.375462
tb_all_cases   0.375462      1.000000 

              hiv_cases  tb_all_cases
hiv_cases      1.000000      0.408971
tb_all_cases   0.408971      1.000000 

              hiv_cases  tb_all_cases
hiv_cases      1.000000      0.400433
tb_all_cases   0.400433      1.000000 

              hiv_cases  tb_all_cases
hiv_cases      1.000000      0.345957
tb_all_cases   0.345957      1.000000 



# GRAPH 2: HIV VS TB (removed outliers)

In [167]:
# removing outliers

# 2015
mean = np.mean(tb2015, axis=0)
sd = np.std(tb2015, axis=0)

mean_hiv = mean.loc['hiv_cases']
mean_tb = mean.loc['tb_all_cases']
sd_hiv = sd.loc['hiv_cases']
sd_tb = sd.loc['tb_all_cases']

tb_2015_no_outlier = tb2015[(tb2015['hiv_cases'] > mean_hiv- 2*sd_hiv) & (tb2015['tb_all_cases'] > mean_tb - 2*sd_tb)][['hiv_cases','tb_all_cases']]
tb_2015_no_outlier = tb_2015_no_outlier[(tb_2015_no_outlier['hiv_cases'] < mean_hiv+ 2*sd_hiv) & (tb_2015_no_outlier['tb_all_cases'] < mean_tb + 2*sd_tb)][['hiv_cases','tb_all_cases']]


# 2010
mean = np.mean(tb2010, axis=0)
sd = np.std(tb2010, axis=0)

mean_hiv = mean.loc['hiv_cases']
mean_tb = mean.loc['tb_all_cases']
sd_hiv = sd.loc['hiv_cases']
sd_tb = sd.loc['tb_all_cases']

tb_2010_no_outlier = tb2010[(tb2010['hiv_cases'] > mean_hiv- 2*sd_hiv) & (tb2010['tb_all_cases'] > mean_tb - 2*sd_tb)][['hiv_cases','tb_all_cases']]
tb_2010_no_outlier = tb_2010_no_outlier[(tb_2010_no_outlier['hiv_cases'] < mean_hiv+ 2*sd_hiv) & (tb_2010_no_outlier['tb_all_cases'] < mean_tb + 2*sd_tb)][['hiv_cases','tb_all_cases']]


# 2005
mean = np.mean(tb2005, axis=0)
sd = np.std(tb2005, axis=0)

mean_hiv = mean.loc['hiv_cases']
mean_tb = mean.loc['tb_all_cases']
sd_hiv = sd.loc['hiv_cases']
sd_tb = sd.loc['tb_all_cases']

tb_2005_no_outlier = tb2005[(tb2005['hiv_cases'] > mean_hiv- 2*sd_hiv) & (tb2005['tb_all_cases'] > mean_tb - 2*sd_tb)][['hiv_cases','tb_all_cases']]
tb_2005_no_outlier = tb_2005_no_outlier[(tb_2005_no_outlier['hiv_cases'] < mean_hiv+ 2*sd_hiv) & (tb_2005_no_outlier['tb_all_cases'] < mean_tb + 2*sd_tb)][['hiv_cases','tb_all_cases']]

# 2000
mean = np.mean(tb2000, axis=0)
sd = np.std(tb2000, axis=0)

mean_hiv = mean.loc['hiv_cases']
mean_tb = mean.loc['tb_all_cases']
sd_hiv = sd.loc['hiv_cases']
sd_tb = sd.loc['tb_all_cases']

tb_2000_no_outlier = tb2000[(tb2000['hiv_cases'] > mean_hiv- 2*sd_hiv) & (tb2000['tb_all_cases'] > mean_tb - 2*sd_tb)][['hiv_cases','tb_all_cases']]
tb_2000_no_outlier = tb_2000_no_outlier[(tb_2000_no_outlier['hiv_cases'] < mean_hiv+ 2*sd_hiv) & (tb_2000_no_outlier['tb_all_cases'] < mean_tb + 2*sd_tb)][['hiv_cases','tb_all_cases']]


# Create traces

from scipy import stats
slope, intercept, r_value, p_value, std_err = stats.linregress(tb_2015_no_outlier['hiv_cases'],tb_2015_no_outlier['tb_all_cases'])
line = slope*tb_2015_no_outlier['hiv_cases']+intercept

trace1 = Scatter(
    x=tb_2015_no_outlier['hiv_cases'], y=tb_2015_no_outlier['tb_all_cases'],
    mode='markers',
    name='2015',
    marker=dict(color= 'rgb(0, 255, 255)')
)

trace2 = Scatter(
                  x=tb_2015_no_outlier['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2015Line',
    marker=dict(color= 'rgb(0, 255, 255)')
)


slope, intercept, r_value, p_value, std_err = stats.linregress(tb_2010_no_outlier['hiv_cases'],tb_2010_no_outlier['tb_all_cases'])
line = slope*tb_2010_no_outlier['hiv_cases']+intercept


trace3 = Scatter(
    x=tb_2010_no_outlier['hiv_cases'], y=tb_2010_no_outlier['tb_all_cases'],
    mode='markers',
    name='2010',
    marker=dict(color= 'rgb(250, 128, 114)')
)

trace4 = Scatter(
                  x=tb_2010_no_outlier['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2010Line',
    marker=dict(color= 'rgb(250, 128, 114)')
)

slope, intercept, r_value, p_value, std_err = stats.linregress(tb_2005_no_outlier['hiv_cases'],tb_2005_no_outlier['tb_all_cases'])
line = slope*tb_2005_no_outlier['hiv_cases']+intercept



trace5 = Scatter(
    x=tb_2005_no_outlier['hiv_cases'], y=tb_2005_no_outlier['tb_all_cases'],
    mode='markers',
    name='2005',
    marker=dict(color= 'rgb(142, 69, 133)')
)

trace6 = Scatter(
                  x=tb_2005_no_outlier['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2005Line',
    marker=dict(color= 'rgb(142, 69, 133)')
)

slope, intercept, r_value, p_value, std_err = stats.linregress(tb_2000_no_outlier['hiv_cases'],tb_2000_no_outlier['tb_all_cases'])
line = slope*tb_2000_no_outlier['hiv_cases']+intercept

trace7 = Scatter(
    x=tb_2000_no_outlier['hiv_cases'], y=tb_2000_no_outlier['tb_all_cases'],
    mode='markers',
    name='2000',
    marker=dict(color= 'rgb(218, 165, 32)')
)

trace8 = Scatter(
                  x=tb_2000_no_outlier['hiv_cases'], 
                  y=line, 
                  mode='lines',
    showlegend=False,
    name='2000Line',
    marker=dict(color= 'rgb(218, 165, 32)')
)

data = Data([trace1, trace2, trace3, trace4, trace5, trace6, trace7, trace8])
layout = Layout(
    title='HIV Cases VS Tuberculosis Cases (Removed Outliers). Data source: WHO',
    yaxis=dict(title='Tuberculosis Cases'),
    xaxis=dict(title='HIV Cases'),
    updatemenus=list([
        dict(
            x=-0.05,
            y=1,
            yanchor='top',
            buttons=list([
                dict(
                    args=['visible', [True, True, True, True, True, True, True, True]],
                    label='All',
                    method='restyle'
                ),
                dict(
                    args=['visible', [True, True, False, False, False, False, False, False]],
                    label='2015',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, True, True, False, False, False, False]],
                    label='2000',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, True, True, False, False]],
                    label='2005',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, False, False, True, True]],
                    label='2000',
                    method='restyle'
                )
            ]),
        )
    ]),
)
fig = Figure(data=data, layout=layout)
py.iplot(fig)

# GRAPH 3: TB cases in top 15 HIV countries

In [99]:
# data sorted by hiv descending

tb2015_hivsort = tb2015.sort_values('hiv_cases', ascending=False)
tb2010_hivsort = tb2010.sort_values('hiv_cases', ascending=False)
tb2005_hivsort = tb2005.sort_values('hiv_cases', ascending=False)
tb2000_hivsort = tb2000.sort_values('hiv_cases', ascending=False)

In [175]:
trace1 = go.Bar(
    y=tb2015_hivsort['country'][:15][::-1],
    x=tb2015_hivsort['hiv_cases'][:15][::-1],
    name='Hiv count 2015',
    orientation='h'
)
trace2 = go.Bar(
    y=tb2015_hivsort['country'][:15][::-1],
    x=tb2015_hivsort['tb_all_cases'][:15][::-1],
    name='Tb count 2015',
    orientation='h'
)

trace3 = go.Bar(
    y=tb2010_hivsort['country'][:15][::-1],
    x=tb2010_hivsort['hiv_cases'][:15][::-1],
    name='Hiv count 2010',
    orientation='h',
    visible=False
)
trace4 = go.Bar(
    y=tb2010_hivsort['country'][:15][::-1],
    x=tb2010_hivsort['tb_all_cases'][:15][::-1],
    name='Tb count 2010',
    orientation='h',
    visible=False
)

trace5 = go.Bar(
    y=tb2005_hivsort['country'][:15][::-1],
    x=tb2005_hivsort['hiv_cases'][:15][::-1],
    name='Hiv count 2005',
    orientation='h',
    visible=False
)
trace6 = go.Bar(
    y=tb2005_hivsort['country'][:15][::-1],
    x=tb2005_hivsort['tb_all_cases'][:15][::-1],
    name='Tb count 2005',
    orientation='h',
    visible=False
)

trace7 = go.Bar(
    y=tb2000_hivsort['country'][:15][::-1],
    x=tb2000_hivsort['hiv_cases'][:15][::-1],
    name='Hiv count 2000',
    orientation='h',
    visible=False
)
trace8 = go.Bar(
    y=tb2000_hivsort['country'][:15][::-1],
    x=tb2000_hivsort['tb_all_cases'][:15][::-1],
    name='Tb count 2000',
    orientation='h',
    visible=False
)

data = [trace2, trace1, trace4, trace3, trace6, trace5, trace8, trace7]
layout = go.Layout(
    title='HIV and TB Case Comparison (top 10 highest HIV countries). Data source: WHO',
    xaxis=dict(
        title='Cases',
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    yaxis=dict(
        title='Country',
        titlefont=dict(
            size=16,
            color='rgb(107, 107, 107)'
        ),
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    barmode='group',
    bargap=0.15,
    bargroupgap=0.1,
    updatemenus=list([
        dict(
            x=-0.15,
            y=1,
            yanchor='top',
            buttons=list([
                dict(
                    args=['visible', [True, True, False, False, False, False, False, False]],
                    label='2015',
                    method='restyle'
                ),
                dict(
                    args=['visible',[False, False, True, True, False, False, False, False]],
                    label='2010',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, True, True, False, False]],
                    label='2005',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, False, False, True, True]],
                    label='2000',
                    method='restyle'
                )
            ]),
        )
    ])
)

fig = go.Figure(data=data, layout=layout)
py.iplot(fig, filename='style-bar')

# GRAPH 4: TB cases in bottom 15 HIV countries

In [176]:
trace1 = go.Bar(
    y=tb2015_hivsort['country'][-15:][::-1],
    x=tb2015_hivsort['hiv_cases'][-15:][::-1],
    name='Hiv count 2015',
    orientation='h'
)
trace2 = go.Bar(
    y=tb2015_hivsort['country'][-15:][::-1],
    x=tb2015_hivsort['tb_all_cases'][-15:][::-1],
    name='Tb count 2015',
    orientation='h'
)

trace3 = go.Bar(
    y=tb2010_hivsort['country'][-15:][::-1],
    x=tb2010_hivsort['hiv_cases'][-15:][::-1],
    name='Hiv count 2010',
    orientation='h',
    visible=False
)
trace4 = go.Bar(
    y=tb2010_hivsort['country'][-15:][::-1],
    x=tb2010_hivsort['tb_all_cases'][-15:][::-1],
    name='Tb count 2010',
    orientation='h',
    visible=False
)

trace5 = go.Bar(
    y=tb2005_hivsort['country'][-15:][::-1],
    x=tb2005_hivsort['hiv_cases'][-15:][::-1],
    name='Hiv count 2005',
    orientation='h',
    visible=False
)
trace6 = go.Bar(
    y=tb2005_hivsort['country'][-15:][::-1],
    x=tb2005_hivsort['tb_all_cases'][-15:][::-1],
    name='Tb count 2005',
    orientation='h',
    visible=False
)

trace7 = go.Bar(
    y=tb2000_hivsort['country'][-15:][::-1],
    x=tb2000_hivsort['hiv_cases'][-15:][::-1],
    name='Hiv count 2000',
    orientation='h',
    visible=False
)
trace8 = go.Bar(
    y=tb2000_hivsort['country'][-15:][::-1],
    x=tb2000_hivsort['tb_all_cases'][-15:][::-1],
    name='Tb count 2000',
    orientation='h',
    visible=False
)

data = [trace2, trace1, trace4, trace3, trace6, trace5, trace8, trace7]
layout = go.Layout(
    title='HIV and TB Case Comparison (top 10 lowest HIV countries). Data source: WHO',
    xaxis=dict(
        title='Cases',
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    yaxis=dict(
        title='Country',
        titlefont=dict(
            size=16,
            color='rgb(107, 107, 107)'
        ),
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    barmode='group',
    bargap=0.15,
    bargroupgap=0.1,
    updatemenus=list([
        dict(
            x=-0.15,
            y=1,
            yanchor='top',
            buttons=list([
                dict(
                    args=['visible', [True, True, False, False, False, False, False, False]],
                    label='2015',
                    method='restyle'
                ),
                dict(
                    args=['visible',[False, False, True, True, False, False, False, False]],
                    label='2010',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, True, True, False, False]],
                    label='2005',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, False, False, True, True]],
                    label='2000',
                    method='restyle'
                )
            ]),
        )
    ])
)

fig = go.Figure(data=data, layout=layout)
py.iplot(fig, filename='style-bar')

# graph 5: % of tb cases with hiv

In [112]:
trace1 = go.Bar(
    y=tb2015_hivsort['country'][:10][::-1],
    x=tb2015_hivsort['%_tb+hiv_cases'][:10][::-1],
    name='Hiv count 2015',
    orientation='h',
    marker=dict(color='rgb(218, 112, 214)')
)


trace2 = go.Bar(
    y=tb2010_hivsort['country'][:10][::-1],
    x=tb2010_hivsort['%_tb+hiv_cases'][:10][::-1],
    name='Hiv count 2010',
    orientation='h',
    visible=False,
    marker=dict(color='rgb(218, 112, 214)')
)


trace3 = go.Bar(
    y=tb2005_hivsort['country'][:10][::-1],
    x=tb2005_hivsort['%_tb+hiv_cases'][:10][::-1],
    name='Hiv count 2005',
    orientation='h',
    visible=False,
    marker=dict(color='rgb(218, 112, 214)')
)


trace4 = go.Bar(
    y=tb2000_hivsort['country'][:10][::-1],
    x=tb2000_hivsort['%_tb+hiv_cases'][:10][::-1],
    name='Hiv count 2000',
    orientation='h',
    visible=False,
    marker=dict(color='rgb(218, 112, 214)')
)


data = [trace1, trace2, trace3, trace4]
layout = go.Layout(
    title='% of TB cases with HIV (top 10 highest HIV countries). Data source: WHO',
    xaxis=dict(
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    yaxis=dict(
        title='Country',
        titlefont=dict(
            size=16,
            color='rgb(107, 107, 107)'
        ),
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    legend=dict(
        bgcolor='rgba(255, 255, 255, 0)',
        bordercolor='rgba(255, 255, 255, 0)'
    ),
    barmode='group',
    bargap=0.15,
    bargroupgap=0.1,
    updatemenus=list([
        dict(
            x=-0.15,
            y=1,
            yanchor='top',
            buttons=list([
                dict(
                    args=['visible', [True, False, False, False]],
                    label='2015',
                    method='restyle'
                ),
                dict(
                    args=['visible',[False, True, False, False]],
                    label='2010',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, True, False]],
                    label='2005',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, True]],
                    label='2000',
                    method='restyle'
                )
            ]),
        )
    ])
)

fig = go.Figure(data=data, layout=layout)
py.iplot(fig, filename='style-bar')

### all tb cases have hiv involved 

In [119]:
print(sum(tb2015['%_tb+hiv_cases'] == 0))
print(sum(tb2010['%_tb+hiv_cases'] == 0))
print(sum(tb2005['%_tb+hiv_cases'] == 0))
print(sum(tb2000['%_tb+hiv_cases'] == 0))

0
0
0
0


# graph 6: tb+hiv vs only tb by region 

In [245]:
tb2015_group_region = tb2015.groupby('region').mean().reset_index()
tb2010_group_region = tb2015.groupby('region').mean().reset_index()
tb2015_group_region = tb2015.groupby('region').mean().reset_index()
tb2015_group_region = tb2015.groupby('region').mean().reset_index()

,region,year,all_tb,tb_and_hiv,tb_and_hiv_percent,hiv_cases,only_tb
0,AFR,2015.0,48035.000000,17788.225000,28.532000,530955.000000,30246.775000
1,AMR,2015.0,9501.518519,1145.296296,15.466667,73696.296296,8356.222222
2,EMR,2015.0,65486.363636,1179.181818,2.074545,29500.000000,64307.181818
3,EUR,2015.0,9327.857143,952.142857,7.928571,49292.857143,8375.714286
4,SEA,2015.0,656142.857143,32224.714286,5.485714,500400.000000,623918.142857
5,WPR,2015.0,83628.571429,2519.857143,4.367143,76500.000000,81108.714286


# TOPIC 4

In [177]:
columns = ['country', 'iso3', 'year', 'e_mort_tbhiv_num', 'e_mort_exc_tbhiv_num', 
          'e_inc_tbhiv_num', 'e_inc_num']

In [178]:
df = pd.read_csv('basic_TB_data.csv')[columns]

# rename columns for easier readability
df.columns = ['country', 'code', 'year', 'death_tb+hiv', 'death_tb_only',
              'cases_tb+hiv', 'cases_all_form']

df.head()

,country,code,year,death_tb+hiv,death_tb_only,cases_tb+hiv,cases_all_form
0,Afghanistan,AFG,2000,64.0,13000.0,120.0,37000
1,Afghanistan,AFG,2001,65.0,13000.0,140.0,39000
2,Afghanistan,AFG,2002,65.0,12000.0,150.0,41000
3,Afghanistan,AFG,2003,74.0,13000.0,170.0,43000
4,Afghanistan,AFG,2004,72.0,12000.0,190.0,44000


In [179]:
# create new column consisting of people with tuberculosis and no hiv
# cases_total - cases_tb+hiv
df['cases_tb_only'] = df['cases_all_form'] - df['cases_tb+hiv']
df = df.drop('cases_all_form', axis=1)
df.head()

,country,code,year,death_tb+hiv,death_tb_only,cases_tb+hiv,cases_tb_only
0,Afghanistan,AFG,2000,64.0,13000.0,120.0,36880.0
1,Afghanistan,AFG,2001,65.0,13000.0,140.0,38860.0
2,Afghanistan,AFG,2002,65.0,12000.0,150.0,40850.0
3,Afghanistan,AFG,2003,74.0,13000.0,170.0,42830.0
4,Afghanistan,AFG,2004,72.0,12000.0,190.0,43810.0


In [180]:
# create percentages column

# mortality of those that had both tb and hiv
df['%_death_tb+hiv'] = df['death_tb+hiv'] / df['cases_tb+hiv']

# mortality of those that had only tb
df['%_death_tb_only'] = df['death_tb_only'] / df['cases_tb_only']

# drop non-% columns
col_to_drop = [
    'death_tb+hiv', 'cases_tb+hiv', 'death_tb_only', 'cases_tb_only'
]
df = df.drop(col_to_drop, axis=1)

# rounded % columns to 2 decimals and multiply by 100
df = df.round(2)
df[["%_death_tb+hiv","%_death_tb_only"]] *= 100

# remove null rows and change to integer data type
df = df[(df['%_death_tb+hiv'].notnull()) & (df['%_death_tb_only'].notnull())] 
df[["%_death_tb+hiv","%_death_tb_only"]] = df[["%_death_tb+hiv","%_death_tb_only"]].astype(np.int8)

print(len(df))

df.head()

2777


,country,code,year,%_death_tb+hiv,%_death_tb_only
0,Afghanistan,AFG,2000,53,35
1,Afghanistan,AFG,2001,46,33
2,Afghanistan,AFG,2002,43,28
3,Afghanistan,AFG,2003,44,30
4,Afghanistan,AFG,2004,38,27


In [181]:
# group into different years

df2015 = df[df['year'] == 2015]
df2010 = df[df['year'] == 2010]
df2005 = df[df['year'] == 2005]
df2000 = df[df['year'] == 2000]

In [ ]:
# average death count over the years

# bar graph showing deaths of different country

# zoom into top deaths country and do another bar graph

# GRAPH 1: deaths over the years

In [191]:
df_avg_deaths_by_year = df.groupby('year').mean()
# round to two decimals
df_avg_deaths_by_year = df_avg_deaths_by_year.round(2)
df_avg_deaths_by_year = df_avg_deaths_by_year.reset_index()
df_avg_deaths_by_year.head()

,year,%_death_tb+hiv,%_death_tb_only
0,2000,23.13,16.15
1,2001,23.54,15.52
2,2002,23.45,14.52
3,2003,21.97,14.14
4,2004,21.85,14.93


In [201]:
# Create and style traces
trace0 = go.Scatter(
    x = df_avg_deaths_by_year['year'],
    y = df_avg_deaths_by_year['%_death_tb+hiv'],
    name = 'With HIV',
    line = dict(
        color = ('rgb(205, 12, 24)'),
        width = 4)
)
trace1 = go.Scatter(
    x = df_avg_deaths_by_year['year'],
    y = df_avg_deaths_by_year['%_death_tb_only'],
    name = 'Without HIV',
    line = dict(
        color = ('rgb(22, 96, 167)'),
        width = 4,)
)
data = [trace0, trace1]

# Edit the layout
layout = dict(title = 'Tuberculosis Deaths (%) Across The Years. Data source: WHO',
              xaxis = dict(title = 'Year'),
              yaxis = dict(title = '%'),
              )

fig = dict(data=data, layout=layout)
py.iplot(fig, filename='styled-line')

### roughly 8.5% more deaths when HIV is present

In [209]:
df_avg_deaths_by_year = df.groupby('year').mean()
df_avg_deaths_by_year['%_death_tb+hiv'] - df_avg_deaths_by_year['%_death_tb_only']

# across each year

year
2000     6.980892
2001     8.019108
2002     8.930380
2003     7.828221
2004     6.910714
2005     7.655172
2006     8.479769
2007     8.482558
2008     8.316092
2009     8.511111
2010     9.243243
2011     9.601093
2012    10.032609
2013     9.054645
2014     8.848649
2015     9.154696
dtype: float64

In [210]:
t=df_avg_deaths_by_year['%_death_tb+hiv'] - df_avg_deaths_by_year['%_death_tb_only']
sum(t)/len(t)

# across all years averaged total

8.5030594825683412

# GRAPH 2: tuberculosis deaths across countries

In [204]:
trace1 = go.Bar(
    x=df2015['country'],
    y=df2015['%_death_tb+hiv'],
    name='2015 TB+HIV',
)
trace2 = go.Bar(
    x=df2015['country'],
    y=df2015['%_death_tb_only'],
    name='2015 TB only',
)

trace3 = go.Bar(
   x=df2010['country'],
    y=df2010['%_death_tb+hiv'],
    name='2010 TB+HIV',
    visible=False
)
trace4 = go.Bar(
    x=df2010['country'],
    y=df2010['%_death_tb_only'],
    name='2010 TB only',
    visible=False
)

trace5 = go.Bar(
    x=df2005['country'],
    y=df2005['%_death_tb+hiv'],
    name='2005 TB+HIV',
    visible=False
)
trace6 = go.Bar(
    x=df2005['country'],
    y=df2005['%_death_tb_only'],
    name='2005 TB only',
    visible=False
)

trace7 = go.Bar(
    x=df2000['country'],
    y=df2000['%_death_tb+hiv'],
    name='2000 TB+HIV',
    visible=False
)
trace8 = go.Bar(
    x=df2000['country'],
    y=df2000['%_death_tb_only'],
    name='2000 TB only',
    visible=False
)

data = [trace2, trace1, trace4, trace3, trace6, trace5, trace8, trace7]
layout = go.Layout(
    title='Tuberculosis deaths (%). Data source: WHO',
    xaxis=dict(
        title='Country',
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    yaxis=dict(
        title='%',
        titlefont=dict(
            size=16,
            color='rgb(107, 107, 107)'
        ),
        tickfont=dict(
            size=14,
            color='rgb(107, 107, 107)'
        )
    ),
    barmode='stack',
    bargap=0,
    bargroupgap=0,
    updatemenus=list([
        dict(
            x=-0.15,
            y=1,
            yanchor='top',
            buttons=list([
                dict(
                    args=['visible', [True, True, False, False, False, False, False, False]],
                    label='2015',
                    method='restyle'
                ),
                dict(
                    args=['visible',[False, False, True, True, False, False, False, False]],
                    label='2010',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, True, True, False, False]],
                    label='2005',
                    method='restyle'
                ),
                dict(
                    args=['visible', [False, False, False, False, False, False, True, True]],
                    label='2000',
                    method='restyle'
                )
            ]),
        )
    ])
)

fig = go.Figure(data=data, layout=layout)
py.iplot(fig, filename='style-bar')

# GRAPH 3: zoomed in tuberculosis deaths graph for top 15 countries

In [222]:
df2015_sorted_tbhhiv = df2015.sort_values('%_death_tb+hiv', ascending=False)

In [230]:
trace1 = go.Bar(
    y=df2015_sorted_tbhhiv['country'][:15][::-1],
    x=df2015_sorted_tbhhiv['%_death_tb+hiv'][:15][::-1],
    name='TB + HIV',
    orientation = 'h',
)
                           
trace2 = go.Bar(
    y=df2015_sorted_tbhhiv['country'][:15][::-1],
    x=df2015_sorted_tbhhiv['%_death_tb_only'][:15][::-1],
    name='TB Only',
    orientation = 'h',
)

data = [trace2, trace1]
layout = go.Layout(
    barmode='group',
    title='Tuberculosis deaths involving HIV 2015 (%). Source: WHO',
    xaxis=dict(
        title='%',
    ),
    yaxis=dict(
        title='Country',
    ),
)

fig = go.Figure(data=data, layout=layout)
py.iplot(fig, filename='marker-h-bar')